In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
import textwrap
import matplotlib.patches as mpatches
from transformers import AutoTokenizer
from datasets import Dataset
from src.utils import clean_text
import en_core_sci_sm
from dotenv import load_dotenv
import os
import numpy as np

load_dotenv()
nlp_core = en_core_sci_sm.load()

plot_save_dir_path = "./plots_hu"
os.makedirs(plot_save_dir_path, exist_ok=True)


### Fájlútvonalak beállítása (Set file paths)

In [ ]:
dataset_paths = {
                "Nyers":
                    {
                        "Egyesített":"../data/processed/base_dataset.parquet",
                        "Szekciókra szűrt":"../data/processed/section_filtered_dataset.parquet"
                    },
                "Gyakori főcsoportok":{
                    "Normál": 
                        {
                            "Alap": "../data/processed/frequent_chapter/without_dropped_sections/base_dataset.parquet",
                            "Tisztított alap (ModernBERT)": "../data/processed/frequent_chapter/without_dropped_sections/modernbert_cleaned_base_dataset.parquet",
                            "Tisztított alap (TF-IDF LR)" :"../data/processed/frequent_chapter/without_dropped_sections/tfidf_cleaned_base_dataset.parquet",
                            "TSO": "../data/processed/frequent_chapter/without_dropped_sections/t_s_dataset.parquet",
                            "Tisztított TSO (ModernBERT)": "../data/processed/frequent_chapter/without_dropped_sections/modernbert_cleaned_t_s_dataset.parquet",
                            "Tisztított TSO (TF-IDF LR)": "../data/processed/frequent_chapter/without_dropped_sections/tfidf_cleaned_t_s_dataset.parquet"
                        },
                    "Záródiagnózis nélküli":
                        {
                            "Alap": "../data/processed/frequent_chapter/with_dropped_sections/base_dataset.parquet",
                            "Tisztított alap (ModernBERT)": "../data/processed/frequent_chapter/with_dropped_sections/modernbert_cleaned_base_dataset.parquet",
                            "Tisztított alap (TF-IDF LR)" :"../data/processed/frequent_chapter/with_dropped_sections/tfidf_cleaned_base_dataset.parquet",
                            "TSO": "../data/processed/frequent_chapter/with_dropped_sections/t_s_dataset.parquet",
                            "Tisztított TSO (ModernBERT)": "../data/processed/frequent_chapter/with_dropped_sections/modernbert_cleaned_t_s_dataset.parquet",
                            "Tisztított TSO (TF-IDF LR)": "../data/processed/frequent_chapter/with_dropped_sections/tfidf_cleaned_t_s_dataset.parquet"
                        },
                     },
                "Top 50 kód":{
                    "Normál": 
                        {
                            "Alap": "../data/processed/top_50_code/without_dropped_sections/base_dataset.parquet",
                            "Tisztított alap (ModernBERT)": "../data/processed/top_50_code/without_dropped_sections/modernbert_cleaned_base_dataset.parquet",
                            "Tisztított alap (TF-IDF LinearSVC)" :"../data/processed/top_50_code/without_dropped_sections/tfidf_cleaned_base_dataset.parquet",
                            "TSO": "../data/processed/top_50_code/without_dropped_sections/t_s_dataset.parquet",
                            "Tisztított TSO (ModernBERT)": "../data/processed/top_50_code/without_dropped_sections/modernbert_cleaned_t_s_dataset.parquet",
                            "Tisztított TSO (TF-IDF LinearSVC)": "../data/processed/top_50_code/without_dropped_sections/tfidf_cleaned_t_s_dataset.parquet"
                        },
                    "Záródiagnózis nélküli":
                        {
                            "Alap": "../data/processed/top_50_code/with_dropped_sections/base_dataset.parquet",
                            "Tisztított alap (ModernBERT)": "../data/processed/top_50_code/with_dropped_sections/modernbert_cleaned_base_dataset.parquet",
                            "Tisztított alap (TF-IDF LinearSVC)" :"../data/processed/top_50_code/with_dropped_sections/tfidf_cleaned_base_dataset.parquet",
                            "TSO": "../data/processed/top_50_code/with_dropped_sections/t_s_dataset.parquet",
                            "Tisztított TSO (ModernBERT)": "../data/processed/top_50_code/with_dropped_sections/modernbert_cleaned_t_s_dataset.parquet",
                            "Tisztított TSO (TF-IDF LinearSVC)": "../data/processed/top_50_code/with_dropped_sections/tfidf_cleaned_t_s_dataset.parquet"
                        }
                }
}

### Tokenizáló betöltése (Load tokenizer)

In [ ]:
local_model_path = "../models/base/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(local_model_path)

In [ ]:
def tokenizer_function(texts):
     encoding = tokenizer(texts['text'], add_special_tokens=False)
     lengths = [len(ids) for ids in encoding['input_ids']]
     return {'token_length': lengths}

### Egyesített elemzések (Unified analytics)

#### Szöveg tisztítás tesztelése (Text cleaning test)

In [ ]:
text = """[EXAMPLE]"""

In [ ]:
print(text)

In [ ]:
print(clean_text([text]))

In [ ]:
print(clean_text([text], "tfidf", nlp_core))

#### Adathalmazok szószámainak diagramja (Datasets word counts plot)

In [ ]:
counts_data = []

df = pd.read_parquet(dataset_paths["Nyers"]["Egyesített"], columns=["hadm_id"])
counts_data.append({"dataset": "Egyesített", "count": len(df)})

df = pd.read_parquet(dataset_paths["Nyers"]["Szekciókra szűrt"], columns=["hadm_id"])
counts_data.append({"dataset": "Szekciókra szűrt", "count": len(df)})

df = pd.read_parquet(dataset_paths["Gyakori főcsoportok"]["Normál"]["Alap"], columns=["hadm_id"])
counts_data.append({"dataset": "Gyakori főcsoportok alap", "count": len(df)})

df = pd.read_parquet(dataset_paths["Top 50 kód"]["Normál"]["Alap"], columns=["hadm_id"])
counts_data.append({"dataset": "Top 50 kód alap", "count": len(df)})

counts_df = pd.DataFrame(counts_data)

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 5), dpi=1000)

ax = sns.barplot(
    data=counts_df, 
    x="dataset", 
    y="count", 
    palette="viridis",
    edgecolor="black",
    linewidth=0.5
)

for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3, fontsize=11)

plt.title("Rekordok száma adathalmazonként")
plt.xlabel("Adathalmazok")
plt.ylabel("Rekordok száma")

sns.despine()

ax.xaxis.grid(False) 
plt.tight_layout()

plt.savefig(os.path.join(plot_save_dir_path, "record_counts_per_ds.png"), dpi=1000, bbox_inches="tight")

plt.show()

#### Discharge tábla statisztikák (Discharge table statistics)

In [ ]:
discharge = pd.read_csv(os.getenv("DISCHARGE"))

unique_patients = discharge["subject_id"].nunique()
unique_texts = discharge.shape[0]

avg_text_per_patient = unique_texts / unique_patients

print("Unique patients :", unique_patients)
print("Unique texts :", unique_texts)
print("Avarge text per patient :", avg_text_per_patient)

#### Legjobb modell variációk összehasonlítása (Comparing the best model variations)

In [ ]:
stats = {
    "Gyakori főcsoportok":{
        "TF-IDF LR (Tisztított TSO, küszöb. opt., főcsoport)":{
            "Makro F1": 0.755,
            "Mikro F1": 0.819,
        },
        "ModernBERT (Tisztított TSO, súlyozott + küszöb. opt., főcsoport)":{
            "Makro F1": 0.799,
            "Mikro F1": 0.844,
        }
    },
    "Top 50 kód":{
        "TF-IDF LR (Tisztított TSO, küszöb. opt., kód)":{
            "Makro F1": 0.604,
            "Mikro F1": 0.626,
        },
        "ModernBERT (Tisztított TSO, súlyozott, kód)":{
            "Makro F1": 0.660,
            "Mikro F1": 0.702,
        }
    }
}

data = []
model_variations = []

for feladat, modellek in stats.items():
    for modell, metrikak in modellek.items():
        var_name = modell
        if var_name not in model_variations:
            model_variations.append(var_name)
            
        for metrika, ertek in metrikak.items():
            data.append({
                "Kategória": f"{feladat}\n{metrika}", 
                "Modell variáció": var_name,
                "F1 Érték": ertek
            })

df = pd.DataFrame(data)

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 5), dpi=1000)

categories = df["Kategória"].unique()
x = np.arange(len(categories))
width = 0.40  

palette = sns.color_palette("viridis", 4)
color_map = dict(zip(model_variations, palette))

for i, cat in enumerate(categories):
    cat_data = df[df["Kategória"] == cat]
    
    models = cat_data["Modell variáció"].values
    values = cat_data["F1 Érték"].values
    
    bar1 = ax.bar(x[i] - width/2, values[0], width, color=color_map[models[0]], edgecolor="black", linewidth=0.5)
    bar2 = ax.bar(x[i] + width/2, values[1], width, color=color_map[models[1]], edgecolor="black", linewidth=0.5)
    


ax.set_xticks(x)
ax.set_xticklabels(categories)

handles = [mpatches.Patch(facecolor=color_map[m], edgecolor="black", linewidth=0.5, label=m) for m in model_variations]
ax.legend(
    handles=handles, 
    title="Modell variációk", 
    frameon=True, 
    shadow=True, 
    fontsize=9, 
    title_fontsize=10, 
    loc="upper right"
)

plt.title("A legjobb modell variációk teljesítményének összehasonlítása")
plt.xlabel("Osztályozási feladat és metrika")
plt.ylabel("F1-érték")
plt.ylim(0, 1.0)

sns.despine()
ax.xaxis.grid(False) 
plt.tight_layout()

plt.savefig(os.path.join(plot_save_dir_path, "best_models.png"), dpi=1000, bbox_inches="tight")

plt.show()

## ICD-10-CM gyakori főcsoport adathalmazok statisztikái (ICD-10-CM frequent chapter datasets statistics)

In [ ]:
ds = pd.read_parquet(dataset_paths["Nyers"]["Egyesített"], columns=["chapter"])
df1 = pd.DataFrame({
    "count": ds["chapter"].apply(len),
    "dataset": "Egyesített"
})

ds = pd.read_parquet(dataset_paths["Nyers"]["Szekciókra szűrt"], columns=["chapter"])
df2 = pd.DataFrame({
    "count": ds["chapter"].apply(len),
    "dataset": "Szekciókra szűrt"
})

ds = pd.read_parquet(dataset_paths["Gyakori főcsoportok"]["Normál"]["Alap"], columns=["chapter"])
df3 = pd.DataFrame({
    "count": ds["chapter"].apply(len),
    "dataset": "Gyakori főcsoportok alap"
})

combined_df = pd.concat([df1, df2, df3], axis=0)

stats = combined_df.groupby("dataset")["count"].agg(['mean'])

set_stats = {}
for name in stats.index:
    row = stats.loc[name]
    stat = f"{name} (Átl.: {row['mean']:.2f})"    
    set_stats[name] = stat

combined_df["dataset"] = combined_df["dataset"].map(set_stats)

sns.set_theme(style="whitegrid")

counts = ds["chapter"].apply(len)

plt.figure(figsize=(11, 5.5), dpi=1000)

ax = sns.countplot(
    data=combined_df, 
    x="count", 
    hue="dataset", 
    palette="viridis",
    edgecolor="black",
    linewidth=0.5
)

plt.legend(
    title="Adathalmazok",
    frameon=True,
    shadow=True,
    fontsize=10,          
    title_fontsize=11
)
plt.yscale("log")
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
ax.ticklabel_format(style="plain", axis="y")
plt.title("Főcsoportok számának eloszlása zárójelentésenként, adathalmazok szerint")
plt.xlabel("Főcsoportok száma zárójelentésenként")
plt.ylabel("Zárójelentések száma")

sns.despine()

ax.xaxis.grid(False) 
plt.tight_layout()

plt.savefig(os.path.join(plot_save_dir_path, "chapter_number_per_disch.png"), dpi=1000, bbox_inches="tight")

plt.show()

In [ ]:
ds = pd.read_parquet(dataset_paths["Nyers"]["Egyesített"], columns=["chapter"])
ds = ds.explode("chapter")["chapter"].value_counts()
df1 = pd.DataFrame({
    "chapter": ds.index,
    "count": ds.values,
    "dataset": "Egyesített"
})

ds = pd.read_parquet(dataset_paths["Nyers"]["Szekciókra szűrt"], columns=["chapter"])
ds = ds.explode("chapter")["chapter"].value_counts()
df2 = pd.DataFrame({
    "chapter": ds.index,
    "count": ds.values,
    "dataset": "Szekciókra szűrt"
})

ds = pd.read_parquet(dataset_paths["Gyakori főcsoportok"]["Normál"]["Alap"], columns=["chapter"])
ds = ds.explode("chapter")["chapter"].value_counts()
df3 = pd.DataFrame({
    "chapter": ds.index,
    "count": ds.values,
    "dataset": "Gyakori főcsoportok alap"
})

combined_df = pd.concat([df1, df2, df3], axis=0)

order = combined_df.groupby("chapter")["count"].sum().sort_values(ascending=False).index

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 5), dpi=1000)

ax = sns.barplot(
    data=combined_df,
    y="count",
    x="chapter",
    hue="dataset",
    order=order,
    palette="viridis",
    edgecolor="black",
    linewidth=0.5
)


ax.set_yscale("log")
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
ax.ticklabel_format(style='plain', axis='y')

plt.title("Főcsoportok gyakorisága adathalmazonként")
plt.xlabel("Főcsoportok") 
plt.ylabel("Előfordulások száma")
plt.xticks(rotation=90)
plt.legend(
    title="Adathalmazok",
    frameon=True,
    shadow=True,
    fontsize=10,          
    title_fontsize=11
)
sns.despine()
plt.tight_layout()

plt.savefig(os.path.join(plot_save_dir_path, "chapter_counts_per_ds.png"), dpi=1000, bbox_inches="tight")

plt.show()

In [ ]:
paths = {"Nyers": dataset_paths["Nyers"]}
paths.update(dataset_paths["Gyakori főcsoportok"])

stats_list = []
for group_name, datasets in paths.items():
    for ds_name, path in datasets.items():
        df = pd.read_parquet(path, columns=["text"])
        
        word_counts = df["text"].apply(lambda x: len(str(x).split()))
            
        stats_list.append({
            "group": group_name,
            "name": ds_name,
            "Átlag": word_counts.mean(),
            "Medián": word_counts.median(),
        })

df_stats = pd.DataFrame(stats_list)

df_stats["group"] = pd.Categorical(df_stats["group"])

df_stats["key"] = df_stats["group"].astype(str) + " | " + df_stats["name"]

raw_keys = df_stats[df_stats["group"] == "Nyers"]["key"].tolist()

other_names = df_stats[df_stats["group"] != "Nyers"]["name"].unique()

paired_keys = []
for name in other_names:
    normal_key = f"Normál | {name}"
    no_discharge_key = f"Záródiagnózis nélküli | {name}"
    
    if normal_key in df_stats["key"].values:
        paired_keys.append(normal_key)
    if no_discharge_key in df_stats["key"].values:
        paired_keys.append(no_discharge_key)

unique_order = raw_keys + paired_keys
df_melted = df_stats.melt(
    id_vars=["key", "group", "name"], 
    value_vars=["Átlag", "Medián"],
    var_name="metric",
    value_name="word_count"
)

metric_order = ["Átlag", "Medián"]

palettes = {
    "Nyers": sns.color_palette("Greens")[3:], 
    "Normál": sns.color_palette("Blues")[3:],
    "Záródiagnózis nélküli": sns.color_palette("Purples")[3:]
}

fig, ax = plt.subplots(figsize=(10, 5), dpi=1000) 

sns.barplot(
    data=df_melted,
    x="key",    
    y="word_count",    
    hue="metric",
    hue_order=metric_order,
    order=unique_order,
    ax=ax,
    edgecolor="black",
    linewidth=0.5,
    width=0.95
)


for container_idx, container in enumerate(ax.containers):
    metric_name = metric_order[container_idx]
    
 
    for bar, unique_key in zip(container, unique_order):
        
    
        group_name = df_stats[df_stats["key"] == unique_key]["group"].values[0]
  
        if group_name in palettes:
            color = palettes[group_name][container_idx]
            bar.set_facecolor(color)
            
    labels = [f"{metric_name}" if i == 0 else "" for i in range(len(container.datavalues))]
    
    ax.bar_label(
        container, 
        labels=labels,
        padding=5, 
        fontsize=9,
        color='black',
        rotation=90
    )



clean_labels = [key.split(" | ")[1] for key in unique_order]
wrapped_clean_labels = [textwrap.fill(lbl, 15) for lbl in clean_labels]

ax.set_xticks(range(len(unique_order)))
ax.set_xticklabels(wrapped_clean_labels, fontsize=11, rotation=90)

ax.set_xlabel("Adathalmazok")
ax.set_ylabel("Szavak száma")
ax.set_title("Szavak számának átlaga és mediánja adathalmazonként (Gyakori főcsoportok)")

legend_patches = []
for group_name, colors in palettes.items():
    patch = mpatches.Patch(color=colors[-2], label=group_name)
    legend_patches.append(patch)

ax.legend(
    handles=legend_patches, 
    title="Adathalmaz csoportok", 
    loc="upper right",       
    fontsize=10,          
    title_fontsize=11,
    frameon=True,
    shadow=True
)

sns.despine()
plt.tight_layout()

plt.savefig(os.path.join(plot_save_dir_path, "word_counts_per_chapter_ds.png"), dpi=1000, bbox_inches="tight")

plt.show()

In [ ]:
df1 = pd.read_parquet(dataset_paths["Gyakori főcsoportok"]["Normál"]["Alap"], columns=["text"])
ds1 = Dataset.from_pandas(df1)
ds1 = ds1.map(tokenizer_function, batched=True)
df1 = ds1.to_pandas()
df1["dataset"] = "Alap"

df2 = pd.read_parquet(dataset_paths["Gyakori főcsoportok"]["Normál"]["TSO"], columns=["text"])
ds2 = Dataset.from_pandas(df2)
ds2 = ds2.map(tokenizer_function, batched=True)
df2 = ds2.to_pandas()
df2["dataset"] = "TSO"

combined_df = pd.concat([df1, df2], ignore_index=True)

plt.figure(figsize=(10, 5), dpi=1000)
sns.set_theme(style="whitegrid") 
sns.violinplot(data=combined_df, y="dataset", x="token_length", palette="viridis", cut=0)

ax = plt.gca() 
ax.xaxis.set_major_locator(ticker.MultipleLocator(512))

plt.xlim(left=0)

plt.axvline(x=2048, color='red', linestyle='-', linewidth=1.5, label='Token limit (2048)')

plt.title("Tokenek számának eloszlása a szöveges állományokban adathalmazonként (Gyakori főcsoportok)")
plt.xlabel("Tokenek száma")
plt.ylabel("Adathalmaz")
plt.xticks(rotation=-90)
plt.legend(loc='lower right')

plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()

plt.savefig(os.path.join(plot_save_dir_path, "token_counts_per_chapter_ds.png"), dpi=1000, bbox_inches="tight")

plt.show()

## ICD-10-CM top 50 kód adathalmazok statisztikái (ICD-10-CM top 50 code datasets statistics)

In [ ]:
ds = pd.read_parquet(dataset_paths["Nyers"]["Egyesített"], columns=["icd_code"])
df1 = pd.DataFrame({
    "count": ds["icd_code"].apply(len),
    "dataset": "Egyesített"
})

ds = pd.read_parquet(dataset_paths["Nyers"]["Szekciókra szűrt"], columns=["icd_code"])
df2 = pd.DataFrame({
    "count": ds["icd_code"].apply(len),
    "dataset": "Szekciókra szűrt"
})

ds = pd.read_parquet(dataset_paths["Top 50 kód"]["Normál"]["Alap"], columns=["icd_code"])
df3 = pd.DataFrame({
    "count": ds["icd_code"].apply(len),
    "dataset": "Top 50 kód alap"
})

combined_df = pd.concat([df1, df2, df3], axis=0)

stats = combined_df.groupby("dataset")["count"].agg(['mean', 'median'])

set_stats = {}
for name in stats.index:
    row = stats.loc[name]
    stat = f"{name} (Átl.: {row['mean']:.2f})"  
    set_stats[name] = stat

combined_df["dataset"] = combined_df["dataset"].map(set_stats)

sns.set_theme(style="whitegrid")

counts = ds["icd_code"].apply(len)

plt.figure(figsize=(11, 5.5), dpi=1000)

ax = sns.countplot(
    data=combined_df, 
    x="count", 
    hue="dataset", 
    palette="viridis",
    edgecolor="black",
    linewidth=0.5
)

plt.legend(
    title="Adathalmazok",
    frameon=True,
    shadow=True,
    fontsize=10,          
    title_fontsize=11
)

plt.title("Kódok számának eloszlása zárójelentésenként, adathalmazok szerint")
plt.xlabel("Kódok száma zárójelentésenként")
plt.ylabel("Zárójelentések száma")

sns.despine()

ax.xaxis.grid(False) 
plt.tight_layout()

plt.savefig(os.path.join(plot_save_dir_path, "code_number_per_disch.png"), dpi=1000, bbox_inches="tight")

plt.show()

In [ ]:
ds = pd.read_parquet(dataset_paths["Nyers"]["Egyesített"], columns=["icd_code"])
ds = ds.explode("icd_code")["icd_code"].value_counts()
df1 = pd.DataFrame({
    "icd_code": ds.index,
    "count": ds.values,
    "dataset": "Egyesített"
})

ds = pd.read_parquet(dataset_paths["Nyers"]["Szekciókra szűrt"], columns=["icd_code"])
ds = ds.explode("icd_code")["icd_code"].value_counts()
df2 = pd.DataFrame({
    "icd_code": ds.index,
    "count": ds.values,
    "dataset": "Szekciókra szűrt"
})

ds = pd.read_parquet(dataset_paths["Top 50 kód"]["Normál"]["Alap"], columns=["icd_code"])
ds = ds.explode("icd_code")["icd_code"].value_counts()
df3 = pd.DataFrame({
    "icd_code": ds.index,
    "count": ds.values,
    "dataset": "Top 50 kód alap"
})

combined_df = pd.concat([df1, df2, df3], axis=0)
combined_df = combined_df[combined_df["icd_code"].isin(ds.index)]

order = combined_df.groupby("icd_code")["count"].sum().sort_values(ascending=False).index

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 5), dpi=1000)

ax = sns.barplot(
    data=combined_df,
    y="count",
    x="icd_code",
    hue="dataset",
    order=order,
    palette="viridis",
    edgecolor="black",
    linewidth=0.5
)

plt.title("Kódok gyakorisága adathalmazonként")
plt.xlabel("Kódok")
plt.ylabel("Előfordulások száma")
plt.xticks(rotation=90)
plt.legend(
    title="Adathalmazok",
    frameon=True,
    shadow=True,
    fontsize=10,          
    title_fontsize=11
)
sns.despine()
plt.tight_layout()

plt.savefig(os.path.join(plot_save_dir_path, "code_counts_per_ds.png"), dpi=1000, bbox_inches="tight")

plt.show()

In [ ]:
paths = {"Nyers": dataset_paths["Nyers"]}
paths.update(dataset_paths["Top 50 kód"])

stats_list = []
for group_name, datasets in paths.items():
    for ds_name, path in datasets.items():
        df = pd.read_parquet(path, columns=["text"])
        
        word_counts = df["text"].apply(lambda x: len(str(x).split()))
            
        stats_list.append({
            "group": group_name,
            "name": ds_name,
            "Átlag": word_counts.mean(),
            "Medián": word_counts.median(),
        })

df_stats = pd.DataFrame(stats_list)

df_stats["group"] = pd.Categorical(df_stats["group"])

df_stats["key"] = df_stats["group"].astype(str) + " | " + df_stats["name"]

raw_keys = df_stats[df_stats["group"] == "Nyers"]["key"].tolist()

other_names = df_stats[df_stats["group"] != "Nyers"]["name"].unique()

paired_keys = []
for name in other_names:
    normal_key = f"Normál | {name}"
    no_discharge_key = f"Záródiagnózis nélküli | {name}"
    
    if normal_key in df_stats["key"].values:
        paired_keys.append(normal_key)
    if no_discharge_key in df_stats["key"].values:
        paired_keys.append(no_discharge_key)

unique_order = raw_keys + paired_keys
df_melted = df_stats.melt(
    id_vars=["key", "group", "name"], 
    value_vars=["Átlag", "Medián"],
    var_name="metric",
    value_name="word_count"
)

metric_order = ["Átlag", "Medián"]

palettes = {
    "Nyers": sns.color_palette("Greens")[3:], 
    "Normál": sns.color_palette("Blues")[3:],
    "Záródiagnózis nélküli": sns.color_palette("Purples")[3:]
}

fig, ax = plt.subplots(figsize=(10, 5), dpi=1000) 

sns.barplot(
    data=df_melted,
    x="key",    
    y="word_count",    
    hue="metric",
    hue_order=metric_order,
    order=unique_order,
    ax=ax,
    edgecolor="black",
    linewidth=0.5,
    width=0.95
)


for container_idx, container in enumerate(ax.containers):
    metric_name = metric_order[container_idx]
    
 
    for bar, unique_key in zip(container, unique_order):
        
    
        group_name = df_stats[df_stats["key"] == unique_key]["group"].values[0]
  
        if group_name in palettes:
            color = palettes[group_name][container_idx]
            bar.set_facecolor(color)
            
    labels = [f"{metric_name}" if i == 0 else "" for i in range(len(container.datavalues))]
    
    ax.bar_label(
        container, 
        labels=labels,
        padding=5, 
        fontsize=9,
        color='black',
        rotation=90
    )



clean_labels = [key.split(" | ")[1] for key in unique_order]
wrapped_clean_labels = [textwrap.fill(lbl, 15) for lbl in clean_labels]

ax.set_xticks(range(len(unique_order)))
ax.set_xticklabels(wrapped_clean_labels, fontsize=11, rotation=90)

ax.set_xlabel("Adathalmazok")
ax.set_ylabel("Szavak száma")
ax.set_title("Szavak számának átlaga és mediánja adathalmazonként (Top 50 kód)")

legend_patches = []
for group_name, colors in palettes.items():
    patch = mpatches.Patch(color=colors[-2], label=group_name)
    legend_patches.append(patch)

ax.legend(
    handles=legend_patches, 
    title="Adathalmaz csoportok",  
    loc="upper right",       
    fontsize=10,          
    title_fontsize=11,
    frameon=True,
    shadow=True
)

sns.despine()
plt.tight_layout()

plt.savefig(os.path.join(plot_save_dir_path, "word_counts_per_code_ds.png"), dpi=1000, bbox_inches="tight")

plt.show()

In [ ]:
df1 = pd.read_parquet(dataset_paths["Top 50 kód"]["Normál"]["Alap"], columns=["text"])
ds1 = Dataset.from_pandas(df1)
ds1 = ds1.map(tokenizer_function, batched=True)
df1 = ds1.to_pandas()
df1["dataset"] = "Alap"

df2 = pd.read_parquet(dataset_paths["Top 50 kód"]["Normál"]["TSO"], columns=["text"])
ds2 = Dataset.from_pandas(df2)
ds2 = ds2.map(tokenizer_function, batched=True)
df2 = ds2.to_pandas()
df2["dataset"] = "TSO"

combined_df = pd.concat([df1, df2], ignore_index=True)

plt.figure(figsize=(10, 5), dpi=1000)
sns.set_theme(style="whitegrid") 
sns.violinplot(data=combined_df, y="dataset", x="token_length", palette="viridis", cut=0)

ax = plt.gca() 
ax.xaxis.set_major_locator(ticker.MultipleLocator(512))

plt.xlim(left=0)

plt.axvline(x=2048, color='red', linestyle='-', linewidth=1.5, label='Token limit (2048)')

plt.title("Tokenek számának eloszlása a szöveges állományokban adathalmazonként (Top 50 kód)")
plt.xlabel("Tokenek száma")
plt.ylabel("Adathalmaz")
plt.xticks(rotation=-90)
plt.legend(loc='lower right')

plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()

plt.savefig(os.path.join(plot_save_dir_path, "token_counts_per_code_ds.png"), dpi=1000, bbox_inches="tight")

plt.show()